# Track 2 Phase 2 — V32 Indic Parler + BEATs + pretrained Demucs

This Colab notebook constructs **5,000 aligned speech-enhancement pairs** and fine-tunes one always-on pretrained Demucs enhancer.

For each source clip:

1. the supplied transcript is synthesized as clean speech with [`ai4bharat/indic-parler-tts`](https://huggingface.co/ai4bharat/indic-parler-tts);
2. the BEATs+WavLM boundary checkpoint from the supplied Track-1 notebook estimates event geometry;
3. a fine-tuned BEATs AudioSet model matches each source event to semantically similar, isolated residual noise from the paired validation data;
4. the selected residual is placed at a randomized position and mixed at a controlled SNR;
5. the resulting noisy waveform and the exact TTS waveform form a sample-aligned supervised pair.

The BEATs models, Indic Parler-TTS, transcripts, event boundaries and residual bank are **training-data preparation tools only**. Submission inference uses one fine-tuned waveform-to-waveform Demucs enhancer for every clip. There is no filename rule, test-time profile lookup, classifier routing, raw bypass, or natural/synthetic branch.

> **Rules guard.** The supplied 3,178 transcript IDs appear to correspond to Phase-2 evaluation audio. Training or adapting on evaluation-derived transcripts/features is a different permission from training on released validation pairs. The notebook therefore requires an explicit confirmation flag before using those clips. If the rules do not permit transductive use, point `SOURCE_AUDIO` and `TRANSCRIPTS_JSONL` to a permitted Vaani corpus instead.

All expensive stages are resumable on Google Drive. `RUN_PHASE2=False` by default; inspect development and monitoring metrics before producing a ZIP.


## 0 — Environment and configuration


In [1]:
# Run in a Colab GPU runtime. Keep the installed Torch build.
import importlib.util, importlib.metadata, subprocess, sys

required = {
    "numpy": "numpy", "pandas": "pandas", "soundfile": "soundfile",
    "librosa": "librosa", "scipy": "scipy", "tqdm": "tqdm",
    "huggingface_hub": "huggingface_hub", "transformers": "transformers",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running the full notebook.")


Torch: 2.11.0+cu130 | CUDA: True


In [9]:
from pathlib import Path
import os, io, re, gc, json, math, wave, shutil, random, hashlib, zipfile, unicodedata, time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.ndimage import median_filter
from tqdm.auto import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, snapshot_download

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass

ROOT = Path("/content/drive/MyDrive/PHD/Datathon")
VAL_ROOT = ROOT / "validation"
META_PATH = VAL_ROOT / "validationMetadata.json"
SOURCE_AUDIO = ROOT / "input_data.zip"
TRANSCRIPTS_JSONL = ROOT / "transcripts.jsonl"

OUT = ROOT / "track2_phase2_v32_indicparler_beats_demucs"
LOCAL = Path("/content/track2_phase2_v32_indicparler_beats_demucs")
RUN = OUT / "run_v1"
WORK = LOCAL / "run_v1"
for p in (OUT, LOCAL, RUN, WORK): p.mkdir(parents=True, exist_ok=True)
V31_PREP_CACHE = ROOT / "track2_phase2_v31_dhvaani_beats" / "run_v1"
PREP_CACHE = V31_PREP_CACHE if V31_PREP_CACHE.is_dir() else RUN

SR = 16000
HOP, NFFT, N_MELS = 160, 512, 64
FRAME = HOP / SR
SEED = 517
PSEUDO_PAIRS = 5000
TOP_K_PROFILES = 5
MAX_EVENTS_PER_CLIP = 2
SNR_JITTER_DB = 3.0
NATURAL_DEV_LIMIT = 100
NATURAL_MONITOR_LIMIT = 100
SEGMENT = 51200                 # 3.2 s
BATCH_SIZE = 4
FINETUNE_EPOCHS = 6
FROZEN_EPOCHS = 2
LR_DECODER = 5e-5
LR_FULL = 1e-5
IDENTITY_PROBABILITY = .10
COMPRESSION = .3
CHUNK_SAMPLES = 51200
CHUNK_HOP = 25600
EXPECTED_PHASE2_CLIPS = 3178
RUN_PHASE2 = False

# Set True only after confirming that competition rules permit model fitting on
# transcripts/features derived from the evaluation inputs. Validation permission
# alone does not establish this permission.
CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING = True

PARLER_REPO = "ai4bharat/indic-parler-tts"
PARLER_CODE_REPO = "git+https://github.com/huggingface/parler-tts.git"
PARLER_MAX_NEW_TOKENS = 2048
BEATS_BOUNDARY_CKPT = Path("/content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt")
BEATS_PRETRAINED = Path("/content/drive/MyDrive/Datathon/pretrained/BEATs_iter3_plus_AS2M.pt")
BEATS_CLASSIFIER_REPO = "WeiChihChen/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2"
BEATS_CLASSIFIER_FILE = "BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"
BEATS_CLASSIFIER_SHA256 = "e5815275a04b6885e7b8af63d120b29bffae2cd2225cf4915e1ec6d819d3022c"

V29_ROOT = ROOT / "track2_phase2_v29_snnet"
ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"
ASR_PIN_CANDIDATES = [
    Path("/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json"),
    ROOT / "track2_v6_direct_reconstruction" / "sravaani_revision_pin.json",
]
ASR_PIN = next((p for p in ASR_PIN_CANDIDATES if p.is_file()), ASR_PIN_CANDIDATES[-1])
DEVICE = torch.device("cuda")
IMPLEMENTATION = "v32_indicparler_beats_demucs_pretrained_finetune_v1"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Persistent run:", RUN)
print("Reusable preparation cache:", PREP_CACHE)
print("Source:", SOURCE_AUDIO)
print("Transcripts:", TRANSCRIPTS_JSONL)


Persistent run: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v32_indicparler_beats_demucs/run_v1
Reusable preparation cache: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v31_dhvaani_beats/run_v1
Source: /content/drive/MyDrive/PHD/Datathon/input_data.zip
Transcripts: /content/drive/MyDrive/PHD/Datathon/transcripts.jsonl


## 1 — Exact audio I/O, metrics and submission contract


In [3]:
def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x

def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()

def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')

def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0

def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()

def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()

def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a

def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()

class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()

def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))

def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')

def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=9,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


## 2 — Released validation pairs and stable real-data splits


In [4]:
def wav_index(root):
    if not root.is_dir(): raise FileNotFoundError(root)
    result={}
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower()=='.wav':
            if p.name in result: raise ValueError(f'Duplicate filename in {root}: {p.name}')
            result[p.name]=p
    return result

def filename_group(name):
    # Dataset filenames end the apparent participant/session token with digits,
    # followed by a numeric utterance token. Retain prefix before that token.
    m=re.match(r'^(.*?\d{6,})_\d+_',Path(name).stem)
    return m.group(1) if m else Path(name).stem

def make_validation_tables(meta_path,root):
    records=json.loads(meta_path.read_text(encoding='utf-8'))
    if not isinstance(records,list): raise ValueError('Expected a JSON list')
    names=[r['segmentFileName'] for r in records]
    if len(names)!=len(set(names)): raise ValueError('Duplicate metadata filenames')
    noisy=wav_index(root/'syntheticNoiseAudio')
    clean=wav_index(root/'syntheticCleanRefAudio')
    natural=wav_index(root/'naturalNoisyAudio')
    pairs=[]; nat=[]
    for r in tqdm(records,desc='Audit validation audio'):
        name=r['segmentFileName']; cid=Path(name).stem
        if Path(name).name!=name: raise ValueError('Expected basename in segmentFileName')
        if not isinstance(r['syntheticData'],bool): raise ValueError('syntheticData must be Boolean')
        common=dict(clip_id=cid,transcript=r.get('transcript',''),language=r.get('language',''),
                    events=r.get('NoiseSubCategoryTimeStamp',[]))
        if r['syntheticData']:
            if name not in noisy or name not in clean: raise FileNotFoundError(f'Missing pair: {name}')
            x=read_wave_strict(noisy[name]); s=read_wave_strict(clean[name])
            if x.shape!=s.shape: raise ValueError(f'Pair length mismatch: {name}')
            if np.array_equal(x,s): raise ValueError(f'Identical noisy/clean pair: {name}')
            if np.mean(s.astype(np.float64)**2)<1e-10: raise ValueError(f'Silent clean target: {name}')
            pairs.append(dict(**common,noisy_path=str(noisy[name]),clean_path=str(clean[name]),
                group_proxy=filename_group(name),frames=len(x),
                noisy_hash=hashlib.sha256(x.tobytes()).hexdigest(),
                clean_hash=hashlib.sha256(s.tobytes()).hexdigest()))
        else:
            if name not in natural: raise FileNotFoundError(f'Missing natural WAV: {name}')
            x=read_wave_strict(natural[name])
            nat.append(dict(**common,noisy_path=str(natural[name]),frames=len(x),
                            noisy_hash=hashlib.sha256(x.tobytes()).hexdigest()))
    # Union grouping proxies and exact clean/noisy duplicates, including cross roles.
    parent=list(range(len(pairs)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(a,b): parent[find(a)]=find(b)
    seen={}
    for i,r in enumerate(pairs):
        for key in [('proxy',r['group_proxy']),('audio',r['clean_hash']),('audio',r['noisy_hash'])]:
            if key in seen: union(i,seen[key])
            else: seen[key]=i
    groups={}
    for i in range(len(pairs)): groups.setdefault(find(i),[]).append(i)
    if len(groups)<10: raise ValueError('Too few independent groups for a useful split')
    ordered=sorted(groups.values(),key=lambda ids: hashlib.sha256(
        (str(SEED)+'|'+min(pairs[i]['clip_id'] for i in ids)).encode()).hexdigest())
    ng=len(ordered); n_hold=max(1,round(.15*ng)); n_dev=max(1,round(.15*ng))
    for j,ids in enumerate(ordered):
        split='holdout' if j<n_hold else 'dev' if j<n_hold+n_dev else 'train'
        group=min(pairs[i]['clip_id'] for i in ids)
        for i in ids: pairs[i].update(split=split,split_group=group)
    return pd.DataFrame(pairs),pd.DataFrame(nat)

PAIRS,NATURAL=make_validation_tables(META_PATH,VAL_ROOT)
if len(PAIRS)!=695 or len(NATURAL)!=1806:
    raise ValueError(f'Metadata changed: {len(PAIRS)} paired / {len(NATURAL)} natural. Inspect before proceeding.')
DATA_SHA=sha_json(dict(metadata=sha_file(META_PATH),audio=PAIRS[
    ['clip_id','noisy_hash','clean_hash','split']].to_dict('records')))

def natural_sample(df,exclude_ids=(),limit=None):
    rows=df[~df.clip_id.isin(exclude_ids)].copy()
    rows['_key']=rows.clip_id.map(lambda s:hashlib.sha256((str(SEED)+s).encode()).hexdigest())
    rows=rows.sort_values('_key')
    return rows if limit is None else rows.head(limit)
# Prior V27 natural sample is explicitly the development set, not a new holdout.
NAT_DEV=natural_sample(NATURAL,limit=NATURAL_DEV_LIMIT)
NAT_MON=natural_sample(NATURAL,exclude_ids=set(NAT_DEV.clip_id),limit=NATURAL_MONITOR_LIMIT)
SYN_DEV=PAIRS[PAIRS.split=='dev'].copy()
SYN_MON=PAIRS[PAIRS.split=='holdout'].copy()
TRAIN_ROWS=PAIRS[PAIRS.split=='train'].copy()
assert set(NAT_DEV.clip_id).isdisjoint(NAT_MON.clip_id)
print('Pairs by split:',PAIRS.groupby('split').size().to_dict())
print('Natural development / additional monitoring:',len(NAT_DEV),len(NAT_MON))


Audit validation audio:   0%|          | 0/2501 [00:00<?, ?it/s]

Pairs by split: {'dev': 100, 'holdout': 104, 'train': 491}
Natural development / additional monitoring: 100 100


## 3 — Build the isolated validation residual bank

For a paired validation example, the clean component may differ from the noisy recording by a global gain. The notebook estimates that gain primarily outside annotated events and computes

\[
n(t)=x_{\text{noisy}}(t)-g\,s_{\text{clean}}(t).
\]

Only the event crop of this residual enters the bank. Profiles with extreme target-speech correlation are rejected. Development and monitoring profiles never enter pseudo-pair training.


In [5]:
PROFILE_DIR = PREP_CACHE / "noise_profiles_train"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_MANIFEST = PREP_CACHE / "noise_profiles_train.csv"

def event_dicts(raw):
    if raw is None: return []
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw: return []
        raw = json.loads(raw)
    if isinstance(raw, np.ndarray): raw = raw.tolist()
    out = []
    for e in raw if isinstance(raw, (list, tuple)) else []:
        if not isinstance(e, dict): continue
        start = e.get("start", e.get("onset"))
        end = e.get("end", e.get("offset"))
        if start is None or end is None: continue
        out.append(dict(
            start=float(start), end=float(end),
            category=str(e.get("category", "unknown")),
            tag=str(e.get("tag", "unknown")),
        ))
    return out

def clean_text(text):
    text = re.sub(r"<[^>]*>|\[[^\]]*\]", " ", str(text or ""))
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def estimate_clean_gain(x, s, events):
    keep = np.ones(len(x), dtype=bool)
    for e in events:
        a = max(0, int(e["start"] * SR)); b = min(len(x), int(e["end"] * SR))
        keep[a:b] = False
    if keep.sum() < max(SR // 4, len(x) // 10): keep[:] = True
    den = float(np.dot(s[keep].astype(np.float64), s[keep].astype(np.float64)))
    return float(np.dot(x[keep].astype(np.float64), s[keep].astype(np.float64)) / max(den, 1e-10))

def apply_edge_fade(y, ms=10):
    y = np.asarray(y, np.float32).copy(); n = min(len(y)//2, int(SR*ms/1000))
    if n > 1:
        ramp = np.sin(np.linspace(0, np.pi/2, n, endpoint=True))**2
        y[:n] *= ramp; y[-n:] *= ramp[::-1]
    return y

if PROFILE_MANIFEST.exists():
    PROFILES = pd.read_csv(PROFILE_MANIFEST)
else:
    recs = []
    for r in tqdm(TRAIN_ROWS.to_dict("records"), desc="Extract validation residual profiles"):
        x, s = read_wave_strict(r["noisy_path"]), read_wave_strict(r["clean_path"])
        evs = event_dicts(r.get("events"))
        if not evs: continue
        gain = estimate_clean_gain(x, s, evs)
        residual = x - gain * s
        for j, e in enumerate(evs):
            a = max(0, int(e["start"]*SR)); b = min(len(x), int(e["end"]*SR))
            if b-a < int(.12*SR): continue
            n = apply_edge_fade(residual[a:b])
            sc = (gain*s[a:b]).astype(np.float32)
            nrms = float(np.sqrt(np.mean(n.astype(np.float64)**2)+1e-12))
            srms = float(np.sqrt(np.mean(sc.astype(np.float64)**2)+1e-12))
            corr = float(np.dot(n, sc) / max(np.linalg.norm(n)*np.linalg.norm(sc), 1e-10))
            if nrms < 2e-4 or abs(corr) > .35: continue
            pid = f'{r["clip_id"]}__{j:02d}'
            path = PROFILE_DIR / (pid + ".wav")
            write_pcm(path, n)
            recs.append(dict(
                profile_id=pid, source_clip=r["clip_id"], category=e["category"], tag=e["tag"],
                start=e["start"], end=e["end"], duration=(b-a)/SR, gain=gain,
                noise_rms=nrms, clean_rms=srms,
                snr_db=20*math.log10(max(srms,1e-8)/max(nrms,1e-8)),
                speech_correlation=corr, path=str(path), split=r["split"],
            ))
    PROFILES = pd.DataFrame(recs)
    if len(PROFILES) < 20: raise RuntimeError(f"Only {len(PROFILES)} usable profiles; inspect residual extraction.")
    PROFILES.to_csv(PROFILE_MANIFEST, index=False)

assert set(PROFILES.split) == {"train"}, "Only training-split profiles may enter pseudo training."
print("Profiles:", len(PROFILES))
display(PROFILES.groupby("category").agg(profiles=("profile_id","size"), median_snr=("snr_db","median")).sort_values("profiles",ascending=False))


Profiles: 1588


,profiles,median_snr
category,,
human_non_speech,473,-90.441696
vehicle_traffic,467,-7.449909
animal,389,-7.319389
baby_child,165,-7.383913
phone_signal_alarm,51,-7.427625
singing_music,22,-8.138949
appliance_machine,21,-6.256431


## 4 — Detect source noise-event geometry with the supplied boundary model

This section reproduces the BEATs+WavLM encoder and three-head boundary network from the attached notebook. It requires the compact checkpoint created there. Detected event windows are cached one clip at a time.


In [10]:
if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:
    raise RuntimeError(
        "Rules guard: confirm that source/evaluation-derived transcripts and BEATs features may be used "
        "for model fitting, then set CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING=True."
    )
if not TRANSCRIPTS_JSONL.is_file(): raise FileNotFoundError(TRANSCRIPTS_JSONL)
if not SOURCE_AUDIO.exists(): raise FileNotFoundError(SOURCE_AUDIO)
if not BEATS_BOUNDARY_CKPT.is_file():
    raise FileNotFoundError(f"Run the supplied BEATs boundary notebook first: {BEATS_BOUNDARY_CKPT}")

TRANSCRIPTS = [json.loads(line) for line in TRANSCRIPTS_JSONL.read_text(encoding="utf-8").splitlines() if line.strip()]
if len(TRANSCRIPTS) != len({r["clip_id"] for r in TRANSCRIPTS}): raise ValueError("Duplicate transcript IDs")
TRANSCRIPT_MAP = {str(r["clip_id"]): clean_text(r["text"]) for r in TRANSCRIPTS}
if len(TRANSCRIPT_MAP) != EXPECTED_PHASE2_CLIPS:
    raise ValueError(f"Expected {EXPECTED_PHASE2_CLIPS} transcripts, found {len(TRANSCRIPT_MAP)}")

SOURCE_DIR = LOCAL / "source_audio"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_AUDIO.is_file() and zipfile.is_zipfile(SOURCE_AUDIO):
    with zipfile.ZipFile(SOURCE_AUDIO) as z:
        wav_members = [m for m in z.namelist() if m.lower().endswith(".wav") and "__MACOSX" not in m]
        idx = {Path(m).stem:m for m in wav_members}
        missing = set(TRANSCRIPT_MAP)-set(idx)
        if missing: raise ValueError(f"Source ZIP misses {len(missing)} transcript IDs")
        for cid in tqdm(sorted(TRANSCRIPT_MAP), desc="Extract source audio"):
            dst = SOURCE_DIR/(cid+".wav")
            if not dst.exists():
                dst.write_bytes(z.read(idx[cid]))
elif SOURCE_AUDIO.is_dir():
    idx = {p.stem:p for p in SOURCE_AUDIO.rglob("*.wav")}
    for cid in TRANSCRIPT_MAP:
        if cid not in idx: raise FileNotFoundError(cid)
        dst=SOURCE_DIR/(cid+".wav")
        if not dst.exists(): shutil.copyfile(idx[cid],dst)
else:
    raise FileNotFoundError(SOURCE_AUDIO)

for cid in list(TRANSCRIPT_MAP)[:5]: read_wave_strict(SOURCE_DIR/(cid+".wav"))
print("Source audio ready:", len(list(SOURCE_DIR.glob("*.wav"))))


Extract source audio:   0%|          | 0/3178 [00:00<?, ?it/s]

Source audio ready: 3178


In [11]:
# ============================================================
# BEATs: reliable download + load + smoke test
# ============================================================

import os
import sys
import hashlib
import shutil
import torch

from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BEATS_DIR = Path("/content/drive/MyDrive/Datathon/pretrained")
BEATS_DIR.mkdir(parents=True, exist_ok=True)

BEATS_CKPT = BEATS_DIR / "BEATs_iter3_plus_AS2M.pt"

EXPECTED_SHA256 = (
    "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
)

device = DEVICE

# ------------------------------------------------------------
# Checksum helper
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Download from Hugging Face
# ------------------------------------------------------------
def download_beats_checkpoint():

    from huggingface_hub import hf_hub_download

    mirrors = [
        ("lpepino/beats_ckpts", "BEATs_iter3_plus_AS2M.pt"),
        ("mooneyko/BEATs", "BEATs_iter3_plus_AS2M.pt"),
        ("nsivaku/nithin_checkpoints", "BEATs_iter3_plus_AS2M.pt"),
    ]

    for repo_id, filename in mirrors:

        print(f"\nTrying: {repo_id}")

        try:
            downloaded = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
            )

            size_mb = os.path.getsize(downloaded) / 1024**2
            print(f"Downloaded: {size_mb:.1f} MB")

            actual_hash = sha256_file(downloaded)

            if actual_hash != EXPECTED_SHA256:
                print("Checksum mismatch.")
                print("Expected:", EXPECTED_SHA256)
                print("Got     :", actual_hash)
                continue

            print("Checksum OK.")

            shutil.copy2(downloaded, BEATS_CKPT)

            return

        except Exception as e:
            print(f"Failed: {type(e).__name__}: {e}")

    raise RuntimeError(
        "Could not download BEATs checkpoint from any Hugging Face mirror."
    )


# ------------------------------------------------------------
# Validate existing checkpoint
# ------------------------------------------------------------
valid_checkpoint = False

if BEATS_CKPT.exists():

    size_mb = BEATS_CKPT.stat().st_size / 1024**2

    print(f"Existing BEATs checkpoint: {size_mb:.1f} MB")

    if size_mb > 300:

        actual_hash = sha256_file(BEATS_CKPT)

        if actual_hash == EXPECTED_SHA256:
            print("Existing checkpoint checksum OK.")
            valid_checkpoint = True
        else:
            print("Existing checkpoint checksum incorrect.")

if not valid_checkpoint:

    if BEATS_CKPT.exists():
        BEATS_CKPT.unlink()

    print("\nDownloading BEATs checkpoint...")
    download_beats_checkpoint()


print("\nCheckpoint ready:")
print(BEATS_CKPT)


# ============================================================
# Get official Microsoft BEATs implementation
# ============================================================

UNILM_DIR = "/tmp/unilm"

if not os.path.exists(f"{UNILM_DIR}/beats/BEATs.py"):

    shutil.rmtree(UNILM_DIR, ignore_errors=True)
    subprocess.check_call(["git","clone","--depth","1","https://github.com/microsoft/unilm.git",UNILM_DIR])

sys.path.insert(0, f"{UNILM_DIR}/beats")

from BEATs import BEATs, BEATsConfig


# ============================================================
# Load checkpoint
# ============================================================

try:
    checkpoint = torch.load(
        BEATS_CKPT,
        map_location="cpu",
        weights_only=False,
    )

except TypeError:
    # compatibility with older torch
    checkpoint = torch.load(
        BEATS_CKPT,
        map_location="cpu",
    )


print("\nCheckpoint keys:", checkpoint.keys())


cfg = BEATsConfig(checkpoint["cfg"])

beats_model = BEATs(cfg)

beats_model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

beats_model = beats_model.to(device)
beats_model.eval()

for p in beats_model.parameters():
    p.requires_grad_(False)


# ============================================================
# Smoke test
# ============================================================

test_audio = torch.randn(
    1,
    16000,
    device=device
)

with torch.no_grad():

    beats_features = beats_model.extract_features(
        test_audio
    )[0]


print("\n==============================")
print("BEATs READY")
print("==============================")

print("device       :", device)
print("feature shape:", tuple(beats_features.shape))
print("embedding dim:", beats_features.shape[-1])


# Variables expected by the rest of your notebook
AUDIOSET_ENCODER = beats_model
AE_DIM = beats_features.shape[-1]
AE_NAME = "BEATs"

print(f"\nAudio encoder: {AE_NAME}")
print(f"Feature dim  : {AE_DIM}")


Existing BEATs checkpoint: 344.8 MB
Existing checkpoint checksum OK.

Checkpoint ready:
/content/drive/MyDrive/Datathon/pretrained/BEATs_iter3_plus_AS2M.pt

Checkpoint keys: dict_keys(['cfg', 'model'])


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



BEATs READY
device       : cuda
feature shape: (1, 48, 768)
embedding dim: 768

Audio encoder: BEATs
Feature dim  : 768


In [12]:
# Compatibility aliases for the rest of the notebook
BEATS_MODEL = beats_model
BEATS_DIM = beats_features.shape[-1]

print(f"BEATS_MODEL ready: {BEATS_MODEL is not None}")
print(f"BEATS_DIM: {BEATS_DIM}")


BEATS_MODEL ready: True
BEATS_DIM: 768


In [13]:
from transformers import WavLMModel


def make_valid_attention_mask(lengths, max_len, device, dtype=torch.long):
    # WavLM convention: 1 = valid, 0 = padding
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return (idx < lengths.unsqueeze(1)).to(dtype)


def make_padding_mask(lengths, max_len, device):
    # BEATs convention: False = valid, True = padding
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)


class WavLMBranch(nn.Module):
    def __init__(self, name='microsoft/wavlm-base-plus'):
        super().__init__()
        self.m = WavLMModel.from_pretrained(name)
        for p in self.m.parameters():
            p.requires_grad_(False)
        self.dim = int(self.m.config.hidden_size)
        self.m.eval()

    def forward(self, wav, T_target, wav_lengths=None):
        # model.train() on the parent would otherwise reactivate encoder dropout.
        self.m.eval()
        attention_mask = None
        if wav_lengths is not None:
            attention_mask = make_valid_attention_mask(
                wav_lengths.to(wav.device), wav.shape[1], wav.device
            )
        with torch.no_grad():
            h = self.m(wav, attention_mask=attention_mask).last_hidden_state
        h = F.interpolate(
            h.transpose(1, 2).float(), size=T_target,
            mode='linear', align_corners=False
        ).transpose(1, 2)
        return h


class BEATsBranch(nn.Module):
    def __init__(self, model, dim):
        super().__init__()
        self.m = model
        self.dim = int(dim)
        self.m.eval()

    def forward(self, wav, T_target, wav_lengths=None):
        self.m.eval()
        padding_mask = None
        if wav_lengths is not None:
            padding_mask = make_padding_mask(
                wav_lengths.to(wav.device), wav.shape[1], wav.device
            )
        with torch.no_grad():
            h = self.m.extract_features(wav, padding_mask=padding_mask)[0]
        if h.ndim != 3:
            raise RuntimeError(f'BEATs returned shape {tuple(h.shape)}; expected [B,T,D].')
        h = F.interpolate(
            h.transpose(1, 2).float(), size=T_target,
            mode='linear', align_corners=False
        ).transpose(1, 2)
        return h


print(f'WavLM dim=768 | BEATs dim={BEATS_DIM}')


WavLM dim=768 | BEATs dim=768


In [14]:
class BoundaryModel(nn.Module):
    def __init__(self, n_mels=N_MELS, ch=(32, 64, 128, 128), rnn=192,
                 wavlm_proj=128, beats_proj=128):
        super().__init__()

        blk, cin = [], 1
        for c in ch:
            blk += [
                nn.Conv2d(cin, c, 3, padding=1),
                nn.BatchNorm2d(c),
                nn.ReLU(),
                nn.AvgPool2d((2, 1)),
                nn.Dropout2d(0.1),
            ]
            cin = c
        self.cnn = nn.Sequential(*blk)
        self.mel_feat = cin * (n_mels // (2 ** len(ch)))

        self.wavlm = WavLMBranch()
        self.wavlm_proj = nn.Sequential(
            nn.Linear(self.wavlm.dim, wavlm_proj),
            nn.LayerNorm(wavlm_proj),
            nn.ReLU(),
        )

        self.beats = BEATsBranch(BEATS_MODEL, BEATS_DIM)
        self.beats_proj = nn.Sequential(
            nn.Linear(BEATS_DIM, beats_proj),
            nn.LayerNorm(beats_proj),
            nn.ReLU(),
        )

        feat_dim = self.mel_feat + wavlm_proj + beats_proj
        self.rnn = nn.GRU(
            feat_dim, rnn, num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.2
        )
        self.noise_head = nn.Linear(2 * rnn, 1)
        self.onset_head = nn.Linear(2 * rnn, 1)
        self.offset_head = nn.Linear(2 * rnn, 1)

    def forward(self, x_mel, wav, wav_lengths=None):
        h_mel = self.cnn(x_mel)
        B, C, Fq, T = h_mel.shape
        h_mel = h_mel.permute(0, 3, 1, 2).reshape(B, T, C * Fq)

        h_wavlm = self.wavlm_proj(self.wavlm(wav, T, wav_lengths))
        h_beats = self.beats_proj(self.beats(wav, T, wav_lengths))
        h = torch.cat([h_mel, h_wavlm, h_beats], dim=-1)

        h, _ = self.rnn(h)
        noise_logits = self.noise_head(h).squeeze(-1)
        onset_dist = torch.sigmoid(self.onset_head(h).squeeze(-1))
        offset_dist = torch.sigmoid(self.offset_head(h).squeeze(-1))
        return noise_logits, onset_dist, offset_dist


print('BoundaryModel: mel + WavLM + BEATs -> BiGRU -> 3 heads')


BoundaryModel: mel + WavLM + BEATs -> BiGRU -> 3 heads


In [15]:
HOP, NFFT, N_MELS = 160, 512, 64
FRAME = HOP / SR
mel_fb = librosa.filters.mel(sr=SR, n_fft=NFFT, n_mels=N_MELS).astype(np.float32)

def logmel(y):
    spec = np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP, center=True))**2
    return np.log(mel_fb @ spec + 1e-8).astype(np.float32)


In [16]:
def decode_standard(p, dur, thr=0.85, med_ms=200, min_dur=0.12, merge_gap=0.1):
    m = p >= thr
    k = max(1, int(med_ms/1000/FRAME))
    if k > 1: m = median_filter(m.astype(np.float32), size=k) > 0.5
    ev, s = [], None
    for i, v in enumerate(m):
        if v and s is None: s = i
        elif not v and s is not None: ev.append([s*FRAME, i*FRAME]); s = None
    if s is not None: ev.append([s*FRAME, len(m)*FRAME])
    out = []
    for e in ev:
        if out and e[0]-out[-1][1] <= merge_gap: out[-1][1] = e[1]
        else: out.append(e)
    return [(a, min(b,dur)) for a,b in out if b-a >= min_dur]

def snap_boundaries(candidates, onset_dist, offset_dist, dur,
                    snap_window=0.20, min_dur=0.12):
    snapped = []
    W = int(snap_window / FRAME)
    for on, off in candidates:
        on_f, off_f = int(on / FRAME), int(off / FRAME)
        N = len(onset_dist)
        a, b = max(0, on_f - W), min(N, on_f + W)
        best_on = (a + int(np.argmin(onset_dist[a:b]))) * FRAME if b > a else on
        a, b = max(0, off_f - W), min(N, off_f + W)
        best_off = (a + int(np.argmin(offset_dist[a:b]))) * FRAME if b > a else off
        best_off = min(best_off, dur)
        if best_off - best_on >= min_dur:
            snapped.append((best_on, best_off))
    return snapped

print('decode + snap defined')


decode + snap defined


In [ ]:
# Restart-safe source-state bootstrap.  A Colab runtime can retain the
# downloaded model cells while TRANSCRIPT_MAP is absent because the earlier
# guarded preparation cell was skipped or did not finish.
if "TRANSCRIPT_MAP" not in globals() or "SOURCE_DIR" not in globals():
    if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:
        raise RuntimeError(
            "TRANSCRIPT_MAP is unavailable because the guarded source-preparation cell did not complete. "
            "Only if the rules permit source/evaluation-derived transcripts for model fitting, set "
            "CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING=True and rerun this cell."
        )
    if not TRANSCRIPTS_JSONL.is_file(): raise FileNotFoundError(TRANSCRIPTS_JSONL)
    if not SOURCE_AUDIO.exists(): raise FileNotFoundError(SOURCE_AUDIO)
    TRANSCRIPTS = [json.loads(line) for line in TRANSCRIPTS_JSONL.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(TRANSCRIPTS) != len({str(r["clip_id"]) for r in TRANSCRIPTS}):
        raise ValueError("Duplicate transcript IDs")
    TRANSCRIPT_MAP = {str(r["clip_id"]): clean_text(r["text"]) for r in TRANSCRIPTS}
    if len(TRANSCRIPT_MAP) != EXPECTED_PHASE2_CLIPS:
        raise ValueError(f"Expected {EXPECTED_PHASE2_CLIPS} transcripts, found {len(TRANSCRIPT_MAP)}")
    SOURCE_DIR = LOCAL / "source_audio"
    SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    missing_source = [cid for cid in TRANSCRIPT_MAP if not (SOURCE_DIR/(cid+".wav")).is_file()]
    if missing_source and SOURCE_AUDIO.is_file() and zipfile.is_zipfile(SOURCE_AUDIO):
        with zipfile.ZipFile(SOURCE_AUDIO) as z:
            idx = {Path(m).stem:m for m in z.namelist() if m.lower().endswith(".wav") and "__MACOSX" not in m}
            unavailable = set(missing_source)-set(idx)
            if unavailable: raise ValueError(f"Source ZIP misses {len(unavailable)} transcript IDs")
            for cid in tqdm(missing_source, desc="Resume source-audio extraction"):
                (SOURCE_DIR/(cid+".wav")).write_bytes(z.read(idx[cid]))
    elif missing_source and SOURCE_AUDIO.is_dir():
        idx = {p.stem:p for p in SOURCE_AUDIO.rglob("*.wav")}
        unavailable = set(missing_source)-set(idx)
        if unavailable: raise ValueError(f"Source directory misses {len(unavailable)} transcript IDs")
        for cid in tqdm(missing_source, desc="Resume source-audio copy"):
            shutil.copyfile(idx[cid], SOURCE_DIR/(cid+".wav"))
    missing_source = [cid for cid in TRANSCRIPT_MAP if not (SOURCE_DIR/(cid+".wav")).is_file()]
    if missing_source: raise RuntimeError(f"Source cache still misses {len(missing_source)} clips")
    print("Restart-restored source state:", len(TRANSCRIPT_MAP), "clips")

EVENT_DIR = PREP_CACHE / "source_events"
EVENT_DIR.mkdir(parents=True, exist_ok=True)

boundary = BoundaryModel().to(DEVICE)
try:
    saved = torch.load(BEATS_BOUNDARY_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    saved = torch.load(BEATS_BOUNDARY_CKPT, map_location="cpu")
state = saved.get("model", saved)
missing, unexpected = boundary.load_state_dict(state, strict=False)
bad_missing = [k for k in missing if not (k.startswith("wavlm.m.") or k.startswith("beats.m."))]
if bad_missing or unexpected:
    raise RuntimeError(f"Boundary checkpoint mismatch: missing={bad_missing[:5]} unexpected={unexpected[:5]}")
boundary.eval()
P_BOUNDARY = {"thr": .85, "med_ms": 200, "min_dur": .12, "merge_gap": .10}

@torch.inference_mode()
def detect_events(path):
    y = read_wave_strict(path); dur = len(y)/SR
    mel = logmel(y); mel=(mel-mel.mean())/(mel.std()+1e-5); T=mel.shape[1]
    need=T*HOP; yy=np.pad(y,(0,max(0,need-len(y))))[:need].astype(np.float32)
    xt=torch.from_numpy(mel)[None,None].to(DEVICE)
    wt=torch.from_numpy(yy)[None].to(DEVICE)
    wl=torch.tensor([min(len(y),need)],device=DEVICE)
    with torch.amp.autocast(device_type="cuda", enabled=True):
        pn,po,pf=boundary(xt,wt,wl)
    p=torch.sigmoid(pn[0]).float().cpu().numpy()
    on=po[0].float().cpu().numpy(); off=pf[0].float().cpu().numpy()
    cands=decode_standard(p,dur,**P_BOUNDARY)
    events=snap_boundaries(cands,on,off,dur)
    fallback=False
    if not events:
        # Every clip is known to contain noise; preserve that fact while marking
        # detector failure explicitly. Classification uses the full clip.
        events=[(0.0,dur)]; fallback=True
    return dict(duration=dur,events=[{"start":float(a),"end":float(b)} for a,b in events],fallback=fallback)

for cid in tqdm(sorted(TRANSCRIPT_MAP),desc="BEATs+WavLM source boundaries"):
    out=EVENT_DIR/(cid+".json")
    if out.exists(): continue
    put_json(out,dict(clip_id=cid,**detect_events(SOURCE_DIR/(cid+".wav"))))

event_rows=[json.loads((EVENT_DIR/(cid+".json")).read_text()) for cid in sorted(TRANSCRIPT_MAP)]
print("Detected events:",sum(len(r["events"]) for r in event_rows),"| fallbacks:",sum(r["fallback"] for r in event_rows))

# Release the two large encoders before loading the AudioSet classifier.
boundary.cpu()
del boundary, saved, state, BEATS_MODEL, beats_model, AUDIOSET_ENCODER
gc.collect(); torch.cuda.empty_cache()


config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

BEATs+WavLM source boundaries:   0%|          | 0/3178 [00:00<?, ?it/s]

Detected events: 3178 | fallbacks: 3178


## 5 — Match source events to validation residuals with BEATs AudioSet probabilities

The source event crop still contains narrator speech. To reduce that bias, its matching signature is the positive part of `event_probability − nearby_context_probability`. Residual profiles are already isolated by paired subtraction. Cosine similarity selects the five closest profiles; generation samples among those five rather than hard-coding one lookup.


In [ ]:
# Fine-tuned BEATs classifier. The official BEATs implementation cloned above is reused.
cls_path = Path(hf_hub_download(BEATS_CLASSIFIER_REPO, BEATS_CLASSIFIER_FILE))
if sha_file(cls_path) != BEATS_CLASSIFIER_SHA256:
    raise RuntimeError("BEATs classifier checksum mismatch")
try:
    cls_ckpt=torch.load(cls_path,map_location="cpu",weights_only=False)
except TypeError:
    cls_ckpt=torch.load(cls_path,map_location="cpu")
cls_cfg=BEATsConfig(cls_ckpt["cfg"])
beats_cls=BEATs(cls_cfg); beats_cls.load_state_dict(cls_ckpt["model"],strict=True)
beats_cls.to(DEVICE).eval()
for p in beats_cls.parameters(): p.requires_grad_(False)
LABELS=[cls_ckpt.get("label_dict",{}).get(i,cls_ckpt.get("label_dict",{}).get(str(i),str(i))) for i in range(int(cls_cfg.predictor_class))]

@torch.inference_mode()
def audio_probs(y):
    y=np.asarray(y,np.float32)
    if len(y)<SR: y=np.pad(y,(0,SR-len(y)))
    # Limit classification windows to 10 s for bounded memory.
    if len(y)>10*SR:
        st=(len(y)-10*SR)//2; y=y[st:st+10*SR]
    z=torch.from_numpy(y)[None].to(DEVICE)
    out=beats_cls.extract_features(z)[0]
    if out.ndim!=2 or out.shape[0]!=1: raise RuntimeError(f"Unexpected classifier output {tuple(out.shape)}")
    out=out[0].float()
    if float(out.min())<0 or float(out.max())>1: out=torch.sigmoid(out)
    return out.cpu().numpy().astype(np.float32)

def slice_seconds(y,a,b):
    i=max(0,int(a*SR));j=min(len(y),max(i+1,int(b*SR)))
    return y[i:j]

PROFILE_VEC_PATH=PREP_CACHE/"profile_beats_probabilities.npz"
if PROFILE_VEC_PATH.exists():
    pack=np.load(PROFILE_VEC_PATH,allow_pickle=True)
    profile_ids=pack["ids"].astype(str).tolist(); profile_vecs=pack["vectors"].astype(np.float32)
else:
    profile_ids=[];vectors=[]
    for r in tqdm(PROFILES.to_dict("records"),desc="BEATs validation-noise signatures"):
        profile_ids.append(r["profile_id"]);vectors.append(audio_probs(read_wave_strict(r["path"])))
    profile_vecs=np.stack(vectors)
    np.savez_compressed(PROFILE_VEC_PATH,ids=np.asarray(profile_ids),vectors=profile_vecs)
profile_lookup={r["profile_id"]:r for r in PROFILES.to_dict("records")}
profile_unit=profile_vecs/np.maximum(np.linalg.norm(profile_vecs,axis=1,keepdims=True),1e-8)

def context_audio(y,events,event_index,target_len):
    mask=np.ones(len(y),dtype=bool)
    for e in events:
        mask[max(0,int(e["start"]*SR)):min(len(y),int(e["end"]*SR))]=False
    context=y[mask]
    if len(context)<SR//2: context=y
    if len(context)>target_len:
        st=(len(context)-target_len)//2;context=context[st:st+target_len]
    return context

MATCH_DIR=PREP_CACHE/"source_profile_matches"
MATCH_DIR.mkdir(parents=True,exist_ok=True)
for cid in tqdm(sorted(TRANSCRIPT_MAP),desc="BEATs source-to-profile matching"):
    dst=MATCH_DIR/(cid+".json")
    if dst.exists(): continue
    geom=json.loads((EVENT_DIR/(cid+".json")).read_text())
    y=read_wave_strict(SOURCE_DIR/(cid+".wav")); matched=[]
    for j,e in enumerate(geom["events"][:MAX_EVENTS_PER_CLIP]):
        seg=slice_seconds(y,e["start"],e["end"])
        pe=audio_probs(seg)
        pc=audio_probs(context_audio(y,geom["events"],j,len(seg)))
        signature=np.maximum(pe-pc,0)
        if np.linalg.norm(signature)<1e-5:
            signature=pe.copy()
            # Narrator speech often dominates an event crop. Remove the broad
            # speech posterior only in this fallback signature.
            for k,label in enumerate(LABELS):
                if any(q in str(label).lower() for q in ("speech","conversation","narration","monologue")):
                    signature[k]=0
        signature/=max(np.linalg.norm(signature),1e-8)
        sims=profile_unit@signature
        best=np.argsort(-sims)[:TOP_K_PROFILES]
        top=[dict(profile_id=profile_ids[int(k)],score=float(sims[int(k)]),
                  category=str(profile_lookup[profile_ids[int(k)]]["category"])) for k in best]
        labels=np.argsort(-pe)[:5]
        matched.append(dict(start=e["start"],end=e["end"],profiles=top,
                            top_labels=[dict(label=str(LABELS[int(k)]),prob=float(pe[int(k)])) for k in labels]))
    put_json(dst,dict(clip_id=cid,duration=geom["duration"],fallback=geom["fallback"],events=matched))

matches=[json.loads((MATCH_DIR/(cid+".json")).read_text()) for cid in sorted(TRANSCRIPT_MAP)]
print("Matched source clips:",len(matches))
display(pd.DataFrame([e["profiles"][0] for r in matches for e in r["events"]]).groupby("category").size().sort_values(ascending=False).head(20))

beats_cls.cpu();del beats_cls,cls_ckpt,profile_vecs,profile_unit
gc.collect();torch.cuda.empty_cache()


BEATs_iter3_plus_AS2M_finetuned_on_AS2M_(…): reconstructing file:   0%|          |  0.00B /  363MB            

BEATs_iter3_plus_AS2M_finetuned_on_AS2M_(…): downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATs source-to-profile matching:   0%|          | 0/3178 [00:00<?, ?it/s]

Matched source clips: 3178


,0
category,
human_non_speech,1628
animal,725
baby_child,563
phone_signal_alarm,171
appliance_machine,64
vehicle_traffic,19
singing_music,8


## 6 — Generate clean speech with Indic Parler-TTS

Indic Parler-TTS generates only the clean target. Four deterministic style descriptions provide male/female and pace/pitch diversity while explicitly requesting close, dry, noise-free speech. Noise is added later from the BEATs-selected validation residual profiles. TTS is never used during enhancement inference.


In [ ]:
SCRIPT_RANGES={
    "devanagari":(0x0900,0x097F),"bengali_assamese":(0x0980,0x09FF),"gurmukhi":(0x0A00,0x0A7F),
    "gujarati":(0x0A80,0x0AFF),"odia":(0x0B00,0x0B7F),"tamil":(0x0B80,0x0BFF),
    "telugu":(0x0C00,0x0C7F),"kannada":(0x0C80,0x0CFF),"malayalam":(0x0D00,0x0D7F),
    "meitei":(0xABC0,0xABFF),"latin":(0x0041,0x007A),
}
def dominant_script(text):
    counts={k:0 for k in SCRIPT_RANGES}
    for ch in str(text):
        o=ord(ch)
        for k,(a,b) in SCRIPT_RANGES.items():
            if a<=o<=b and ch.isalpha():counts[k]+=1;break
    return max(counts,key=counts.get) if max(counts.values(),default=0)>0 else "other"

PARLER_DESCRIPTIONS=[
    "A male speaker delivers natural speech at a moderate pace and pitch. The recording is very clear, close, dry, and contains no background noise or reverberation.",
    "A female speaker delivers natural speech at a moderate pace and pitch. The recording is very clear, close, dry, and contains no background noise or reverberation.",
    "A male speaker talks slightly slowly with a balanced low pitch. The voice is intelligible and the recording is clean, close, dry, and noise-free.",
    "A female speaker talks slightly quickly with a balanced high pitch. The voice is intelligible and the recording is clean, close, dry, and noise-free.",
]
TTS_CONFIG_SHA=hashlib.sha256(json.dumps(dict(repo=PARLER_REPO,descriptions=PARLER_DESCRIPTIONS,
    max_new_tokens=PARLER_MAX_NEW_TOKENS),sort_keys=True).encode()).hexdigest()


In [ ]:
token=os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token=userdata.get("HF_TOKEN")
    except Exception:pass
if not token:raise RuntimeError("Accept the Indic Parler-TTS access terms and add HF_TOKEN to Colab Secrets.")

if importlib.util.find_spec("parler_tts") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",PARLER_CODE_REPO])
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer,set_seed

load_kwargs=dict(token=token,torch_dtype=torch.float16)
try:
    tts_model=ParlerTTSForConditionalGeneration.from_pretrained(PARLER_REPO,attn_implementation="sdpa",**load_kwargs)
except Exception:
    tts_model=ParlerTTSForConditionalGeneration.from_pretrained(PARLER_REPO,**load_kwargs)
tts_model.to(DEVICE).eval()
tts_tokenizer=AutoTokenizer.from_pretrained(PARLER_REPO,token=token)
description_tokenizer=AutoTokenizer.from_pretrained(tts_model.config.text_encoder._name_or_path,token=token)
tts_sr=int(tts_model.config.sampling_rate)
print("Indic Parler-TTS ready at",tts_sr,"Hz")

TTS_DIR=RUN/"parler_clean_16k";TTS_DIR.mkdir(parents=True,exist_ok=True)
TTS_STATUS=RUN/"parler_status";TTS_STATUS.mkdir(parents=True,exist_ok=True)

def split_tts_text(text,max_chars=260):
    words=str(text).split();chunks=[];cur=[];n=0
    for w in words:
        if cur and n+1+len(w)>max_chars:chunks.append(" ".join(cur));cur=[];n=0
        cur.append(w);n+=len(w)+1
    if cur:chunks.append(" ".join(cur))
    return chunks

@torch.inference_mode()
def synthesize_parler(text,description,seed):
    pieces=[]
    for j,chunk in enumerate(split_tts_text(text)):
        set_seed(seed+j)
        d=description_tokenizer(description,return_tensors="pt").to(DEVICE)
        p=tts_tokenizer(chunk,return_tensors="pt").to(DEVICE)
        generation=tts_model.generate(input_ids=d.input_ids,attention_mask=d.attention_mask,
            prompt_input_ids=p.input_ids,prompt_attention_mask=p.attention_mask,
            do_sample=True,temperature=1.0,max_new_tokens=PARLER_MAX_NEW_TOKENS)
        a=generation[0].float().cpu().numpy().astype(np.float32)
        if a.ndim!=1 or len(a)<tts_sr//4 or not np.isfinite(a).all():raise RuntimeError("Invalid Parler output")
        pieces.append(a)
    y=np.concatenate([v if i==0 else np.concatenate([np.zeros(int(.08*tts_sr),np.float32),v]) for i,v in enumerate(pieces)])
    if tts_sr!=SR:y=librosa.resample(y,orig_sr=tts_sr,target_sr=SR).astype(np.float32)
    peak=float(np.max(np.abs(y)))
    if peak>.95:y*=.95/peak
    return y

for cid,text in tqdm(sorted(TRANSCRIPT_MAP.items()),desc="Indic Parler clean synthesis"):
    wav=TTS_DIR/(cid+".wav");status=TTS_STATUS/(cid+".json")
    if wav.exists() and status.exists():
        try:cached=json.loads(status.read_text())
        except Exception:cached={}
        if cached.get("ok") and cached.get("tts_config_sha")==TTS_CONFIG_SHA:continue
    k=int(hashlib.sha256((str(SEED)+cid).encode()).hexdigest(),16)%len(PARLER_DESCRIPTIONS)
    try:
        y=synthesize_parler(text,PARLER_DESCRIPTIONS[k],SEED+k);write_pcm(wav,y)
        put_json(status,dict(clip_id=cid,ok=True,script=dominant_script(text),description_id=k,
            frames=len(y),tts_config_sha=TTS_CONFIG_SHA))
    except Exception as exc:
        put_json(status,dict(clip_id=cid,ok=False,script=dominant_script(text),description_id=k,
            error=f"{type(exc).__name__}: {exc}",tts_config_sha=TTS_CONFIG_SHA))

statuses={p.stem:json.loads(p.read_text()) for p in TTS_STATUS.glob("*.json")}
ok=[cid for cid,r in statuses.items() if r.get("ok") and r.get("tts_config_sha")==TTS_CONFIG_SHA and (TTS_DIR/(cid+".wav")).is_file()]
print("Current-config TTS success:",len(ok),"/",len(TRANSCRIPT_MAP))
display(pd.Series([statuses[c]["script"] for c in ok]).value_counts().rename("successful_clips").to_frame())
if len(ok)<800:raise RuntimeError("Fewer than 800 valid clean TTS clips; inspect parler_status before pair generation.")

tts_model.cpu();del tts_model,tts_tokenizer,description_tokenizer
gc.collect();torch.cuda.empty_cache()


## 7 — Generate exactly 5,000 aligned pseudo pairs


In [ ]:
PSEUDO_DIR=RUN/"pseudo_pairs";NOISY_DIR=PSEUDO_DIR/"noisy";CLEAN_DIR=PSEUDO_DIR/"clean"
NOISY_DIR.mkdir(parents=True,exist_ok=True);CLEAN_DIR.mkdir(parents=True,exist_ok=True)
PSEUDO_MANIFEST=RUN/"pseudo_pairs.jsonl"

match_map={p.stem:json.loads(p.read_text()) for p in MATCH_DIR.glob("*.json")}
eligible=sorted(cid for cid in TRANSCRIPT_MAP if (TTS_DIR/(cid+".wav")).exists() and cid in match_map and match_map[cid]["events"])
if not eligible:raise RuntimeError("No eligible TTS/match records")

rng=np.random.default_rng(SEED+9001)
source_order=list(eligible)
while len(source_order)<PSEUDO_PAIRS:
    source_order.extend(rng.permutation(eligible).tolist())
source_order=source_order[:PSEUDO_PAIRS]

def fit_noise(n,max_len,rng):
    if len(n)<=max_len:return n
    st=int(rng.integers(0,len(n)-max_len+1));return n[st:st+max_len]

def mix_event(clean,noise,snr_db,rng):
    noise=fit_noise(noise,len(clean),rng)
    if len(noise)<int(.10*SR):raise ValueError("Noise event too short")
    start=int(rng.integers(0,len(clean)-len(noise)+1)) if len(clean)>len(noise) else 0
    speech=clean[start:start+len(noise)]
    srms=float(np.sqrt(np.mean(speech.astype(np.float64)**2)+1e-10))
    if srms<1e-3:srms=float(np.sqrt(np.mean(clean.astype(np.float64)**2)+1e-10))
    nrms=float(np.sqrt(np.mean(noise.astype(np.float64)**2)+1e-10))
    scale=srms/(max(nrms,1e-8)*10**(snr_db/20))
    return start,noise*scale

records=[]
for i,cid in enumerate(tqdm(source_order,desc="Generate pseudo noisy/clean pairs")):
    pair_id=f"pseudo_{i:05d}_{cid}"
    npth=NOISY_DIR/(pair_id+".wav");cpth=CLEAN_DIR/(pair_id+".wav")
    local_rng=np.random.default_rng(SEED+i*104729)
    clean=read_wave_strict(TTS_DIR/(cid+".wav"));noisy=clean.copy();placed=[]
    source_events=match_map[cid]["events"][:MAX_EVENTS_PER_CLIP]
    for e in source_events:
        options=e["profiles"]
        # Softmax-like rank sampling favours the closest match without making a fixed lookup.
        rank=int(local_rng.choice(len(options),p=np.asarray([.45,.25,.15,.10,.05])[:len(options)]/np.asarray([.45,.25,.15,.10,.05])[:len(options)].sum()))
        choice=options[rank];pr=profile_lookup[choice["profile_id"]]
        noise=read_wave_strict(pr["path"])
        snr=float(np.clip(float(pr["snr_db"])+local_rng.uniform(-SNR_JITTER_DB,SNR_JITTER_DB),-12,18))
        start,scaled=mix_event(clean,noise,snr,local_rng)
        noisy[start:start+len(scaled)]+=scaled
        placed.append(dict(start=start/SR,end=(start+len(scaled))/SR,profile_id=choice["profile_id"],
                           category=choice["category"],match_score=choice["score"],snr_db=snr))
    peak=max(float(np.max(np.abs(noisy))),float(np.max(np.abs(clean))),1e-8)
    joint_gain=min(1.0,.98/peak);noisy*=joint_gain;clean*=joint_gain
    write_pcm(npth,noisy);write_pcm(cpth,clean)
    split="pseudo_val" if int(hashlib.sha256(cid.encode()).hexdigest(),16)%10==0 else "pseudo_train"
    records.append(dict(pair_id=pair_id,source_clip=cid,split=split,noisy_path=str(npth),clean_path=str(cpth),
                        frames=len(clean),events=placed,script=dominant_script(TRANSCRIPT_MAP[cid]),joint_gain=joint_gain))
    if (i+1)%100==0:
        PSEUDO_MANIFEST.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in records),encoding="utf-8")
PSEUDO_MANIFEST.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in records),encoding="utf-8")
PSEUDO=pd.DataFrame(records)
print(PSEUDO.groupby("split").size())
display(PSEUDO.explode("events").head())


## 8 — Pseudo-pair integrity and diversity audit


In [ ]:
audit=[]
for r in tqdm(PSEUDO.sample(min(500,len(PSEUDO)),random_state=SEED).to_dict("records"),desc="Audit pseudo pairs"):
    x=read_wave_strict(r["noisy_path"]);s=read_wave_strict(r["clean_path"])
    if x.shape!=s.shape or np.array_equal(canonical_pcm(x),canonical_pcm(s)):raise RuntimeError(f"Invalid pair {r['pair_id']}")
    audit.append(dict(pair_id=r["pair_id"],duration=len(x)/SR,
                      noisy_rms=float(np.sqrt(np.mean(x.astype(np.float64)**2))),
                      clean_rms=float(np.sqrt(np.mean(s.astype(np.float64)**2))),
                      noisy_si_sdr=si_sdr(x,s),events=len(r["events"])))
AUDIT=pd.DataFrame(audit);AUDIT.to_csv(RUN/"pseudo_pair_audit.csv",index=False)
display(AUDIT.describe())
print("Unique source clips:",PSEUDO.source_clip.nunique(),"| unique profiles:",len({e['profile_id'] for es in PSEUDO.events for e in es}))
if PSEUDO.source_clip.nunique()<int(.8*len(eligible)):raise RuntimeError("Insufficient source diversity")


## 9 — Fine-tune a pretrained always-on Demucs enhancer

The notebook first searches for the compatible domain-adapted Demucs epoch-3 checkpoint produced by the earlier validation experiment. If none is compatible, it loads the official DNS64 pretrained weights. Random initialization is forbidden. The first two epochs update only the decoder; subsequent epochs unfreeze the entire model at a lower learning rate.

The main objective is SI-SDR, with small multi-resolution STFT, normalized waveform and outside-event preservation terms. Ten percent clean-to-clean identity examples teach speech preservation, but inference still invokes the neural model on every clip.


In [ ]:
if importlib.util.find_spec("denoiser") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","denoiser==0.1.5"])
from denoiser import pretrained as demucs_pretrained

model=demucs_pretrained.dns64(pretrained=False)
if model.sample_rate!=SR or model.chin!=1:raise RuntimeError("Demucs configuration is not 16-kHz mono")

def unpack_demucs_state(raw):
    if isinstance(raw,dict) and "model" in raw and isinstance(raw["model"],dict):raw=raw["model"]
    if not isinstance(raw,dict):raise TypeError("Checkpoint is not a state dict")
    if any(str(k).startswith("enhancer.") for k in raw):
        raw={str(k)[len("enhancer."):]:v for k,v in raw.items() if str(k).startswith("enhancer.")}
    return raw

domain_candidates=[]
for root in (ROOT/"track2_phase2_v28_pretrained_wer",ROOT/"track2_phase2_v29_snnet"):
    if root.exists():domain_candidates.extend(root.rglob("epoch_03.pt"))
INIT_CHECKPOINT=None
for candidate in sorted(domain_candidates,key=lambda p:p.stat().st_mtime,reverse=True):
    try:
        raw=torch.load(candidate,map_location="cpu",weights_only=True)
        model.load_state_dict(unpack_demucs_state(raw),strict=True)
        INIT_CHECKPOINT=candidate;break
    except Exception:continue

if INIT_CHECKPOINT is None:
    official=torch.hub.load_state_dict_from_url(demucs_pretrained.DNS_64_URL,
        model_dir=str(RUN/"pretrained_weights"),map_location="cpu",check_hash=True,weights_only=True)
    model.load_state_dict(official,strict=True);del official
    INIT_SOURCE="official_dns64"
else:
    INIT_SOURCE=str(INIT_CHECKPOINT)
print("Demucs initialization:",INIT_SOURCE)
INITIAL_CHECKPOINT=RUN/"demucs_initial.pt"
if not INITIAL_CHECKPOINT.exists():torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},INITIAL_CHECKPOINT)


In [ ]:
def neg_si_sdr_loss(y,s):
    y=y-y.mean(-1,keepdim=True);s=s-s.mean(-1,keepdim=True)
    target=(y*s).sum(-1,keepdim=True)/(s.square().sum(-1,keepdim=True)+1e-8)*s
    error=y-target
    score=10*torch.log10((target.square().sum(-1)+1e-8)/(error.square().sum(-1)+1e-8))
    return -score.mean()/20.0

def multires_stft_loss(y,s):
    total=0
    for size in (256,512,1024):
        win=torch.hann_window(size,device=y.device)
        Y=torch.stft(y,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        S=torch.stft(s,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        total+=(torch.log1p(Y)-torch.log1p(S)).abs().mean()
    return total/3

def enhancement_loss(net,x,s,event_mask):
    y=net(x[:,None])[:,0]
    l_si=neg_si_sdr_loss(y,s)
    l_stft=multires_stft_loss(y,s)
    scale=s.square().mean(-1).sqrt().clamp_min(.01)[:,None]
    l_wave=((y-s)/scale).abs().mean()
    outside=1-event_mask
    preserve=(((y-s).abs()/scale)*outside).sum()/outside.sum().clamp_min(1)
    loss=l_si+.10*l_stft+.05*l_wave+.05*preserve
    return loss,dict(si_sdr=l_si,stft=l_stft,wave=l_wave,preserve=preserve),y

class PairDataset(Dataset):
    def __init__(self,records,training=True):self.records=list(records);self.training=training
    def __len__(self):return len(self.records)
    def __getitem__(self,i):
        r=self.records[i];x=read_wave_strict(r["noisy_path"]);s=read_wave_strict(r["clean_path"])
        if len(x)!=len(s):raise ValueError("Pair length mismatch")
        evs=event_dicts(r.get("events",[]));n=min(len(x),SEGMENT)
        rng=np.random.default_rng(SEED+i+(int(time.time()) if self.training else 0))
        identity=bool(self.training and rng.random()<IDENTITY_PROBABILITY)
        if identity:x=s.copy();evs=[]
        start=int(rng.integers(0,len(x)-n+1)) if self.training and len(x)>n else max(0,(len(x)-n)//2)
        xx=np.pad(x[start:start+n],(0,SEGMENT-n));ss=np.pad(s[start:start+n],(0,SEGMENT-n))
        mask=np.zeros(SEGMENT,np.float32)
        for e in evs:
            a=max(0,int(e["start"]*SR)-start);b=min(n,int(e["end"]*SR)-start)
            if b>a:mask[a:b]=1
        return torch.from_numpy(xx.astype(np.float32)),torch.from_numpy(ss.astype(np.float32)),torch.from_numpy(mask)

pseudo_records=PSEUDO[PSEUDO.split=="pseudo_train"].to_dict("records")
real_records=[]
for r in TRAIN_ROWS.to_dict("records"):
    real_records.append(dict(pair_id="real_"+r["clip_id"],noisy_path=r["noisy_path"],clean_path=r["clean_path"],events=r["events"]))
# Repeat genuine pairs so roughly one quarter of updates remain grounded in real speech.
train_records=pseudo_records+real_records*3
val_records=PSEUDO[PSEUDO.split=="pseudo_val"].to_dict("records")
train_ds=PairDataset(train_records,True);val_ds=PairDataset(val_records,False)
train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
print("Training records:",len(train_ds),"| pseudo validation:",len(val_ds),"| identity probability:",IDENTITY_PROBABILITY)


In [ ]:
history_path=RUN/"demucs_training_history.csv"
history=pd.read_csv(history_path).to_dict("records") if history_path.exists() else []
existing=sorted(RUN.glob("demucs_epoch_*.pt"))
start_epoch=0
if existing:
    latest=existing[-1];model.load_state_dict(torch.load(latest,map_location="cpu",weights_only=True),strict=True)
    start_epoch=int(latest.stem.rsplit("_",1)[-1]);print("Resuming after epoch",start_epoch)

def configure_stage(epoch):
    frozen=epoch<=FROZEN_EPOCHS
    for p in model.parameters():p.requires_grad_(True)
    if frozen:
        for p in model.encoder.parameters():p.requires_grad_(False)
        for p in model.lstm.parameters():p.requires_grad_(False)
    lr=LR_DECODER if frozen else LR_FULL
    return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-5),("decoder" if frozen else "full"),lr

def flatten_rnns(net):
    for module in net.modules():
        if isinstance(module,nn.RNNBase):module.flatten_parameters()

@torch.inference_mode()
def validation_loss():
    model.eval();flatten_rnns(model);vals=[]
    for x,s,m in val_dl:
        x,s,m=x.to(DEVICE),s.to(DEVICE),m.to(DEVICE)
        with torch.amp.autocast("cuda",enabled=True):vals.append(float(enhancement_loss(model,x,s,m)[0]))
    return float(np.mean(vals))

CANDIDATES={"demucs_initial":INITIAL_CHECKPOINT}
model.to(DEVICE)
for epoch in range(start_epoch+1,FINETUNE_EPOCHS+1):
    optimizer,stage,lr=configure_stage(epoch)
    model.train();flatten_rnns(model);values=[];parts_sum=[]
    for x,s,m in tqdm(train_dl,desc=f"Demucs fine-tune {epoch}/{FINETUNE_EPOCHS}"):
        x,s,m=x.to(DEVICE,non_blocking=True),s.to(DEVICE,non_blocking=True),m.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=True):loss,parts,_=enhancement_loss(model,x,s,m)
        if not torch.isfinite(loss):raise RuntimeError("Non-finite training loss")
        loss.backward();nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.,error_if_nonfinite=True);optimizer.step()
        values.append(float(loss.detach()));parts_sum.append([float(parts[k].detach()) for k in ("si_sdr","stft","wave","preserve")])
    val=validation_loss();means=np.mean(parts_sum,axis=0)
    rec=dict(epoch=epoch,stage=stage,lr=lr,train_loss=float(np.mean(values)),val_loss=val,
             si_sdr=float(means[0]),stft=float(means[1]),wave=float(means[2]),preserve=float(means[3]))
    history.append(rec);print(rec)
    path=RUN/f"demucs_epoch_{epoch:02d}.pt";tmp=path.with_suffix(".tmp")
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},tmp);os.replace(tmp,path)
    pd.DataFrame(history).to_csv(history_path,index=False);del optimizer
for p in sorted(RUN.glob("demucs_epoch_*.pt")):CANDIDATES[p.stem]=p
model.cpu();gc.collect();torch.cuda.empty_cache()
print("Candidate checkpoints:",list(CANDIDATES))


## 10 — Uniform inference and frozen SraVaani evaluation


In [ ]:
LAST_MODEL_CALLS=0
@torch.inference_mode()
def enhance_audio(y):
    global LAST_MODEL_CALLS
    y=np.asarray(y,np.float32);out=np.zeros(len(y),np.float64);den=np.zeros(len(y),np.float64)
    weight=np.maximum(np.hanning(CHUNK_SAMPLES),1e-3);LAST_MODEL_CALLS=0
    for start in range(0,len(y),CHUNK_HOP):
        count=min(CHUNK_SAMPLES,len(y)-start);seg=np.pad(y[start:start+count],(0,CHUNK_SAMPLES-count))
        xt=torch.from_numpy(seg)[None].to(DEVICE)
        z=model(xt[:,None])[:,0][0].float().cpu().numpy()
        if not np.isfinite(z).all():raise RuntimeError("Invalid enhancement; no fallback")
        out[start:start+count]+=z[:count]*weight[:count];den[start:start+count]+=weight[:count];LAST_MODEL_CALLS+=1
        if start+CHUNK_SAMPLES>=len(y):break
    if (den<=0).any():raise RuntimeError("Uncovered samples")
    return (out/den).astype(np.float32)

def activate(path,device=DEVICE):
    model.load_state_dict(torch.load(path,map_location="cpu",weights_only=True),strict=True)
    model.to(device).eval();flatten_rnns(model)


In [ ]:
from transformers import AutoModel
pin=json.loads(ASR_PIN.read_text()); ASR_REV=str(pin['revision'])
if pin.get('repo')!=ASR_REPO or not re.fullmatch('[0-9a-fA-F]{40}',ASR_REV):
    raise ValueError('Invalid existing SraVaani revision pin')
asr_identity=dict(repo=ASR_REPO,revision=ASR_REV,task='transcribe_return_hypotheses',
    pipeline='v29_content_filename_nograd_v1',torch=str(torch.__version__),
    transformers=importlib.metadata.version('transformers'))
ASR_SIGNATURE=sha_json(asr_identity)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:pass
asr_model=AutoModel.from_pretrained(ASR_REPO,revision=ASR_REV,code_revision=ASR_REV,
    trust_remote_code=True,token=token).to(DEVICE).eval()
put_json(RUN/'asr_identity.json',asr_identity)

class FreshASR:
    def __init__(self,model):
        self.model=model;self.cache=ROOT/'track2_phase2_v28_pretrained_wer'/'asr_cache_v28'/ASR_SIGNATURE
        self.cache.mkdir(parents=True,exist_ok=True)
        self.stage=WORK/'asr_inputs';self.stage.mkdir(exist_ok=True)
    def transcribe(self,paths,force=False):
        result={}
        for cid,p in tqdm(paths.items(),desc='Content-audited SraVaani'):
            p=Path(p);h=sha_file(p);dest=self.cache/(h+'.json')
            rec=None
            if dest.exists() and not force:
                trial=json.loads(dest.read_text())
                if trial.get('wav_sha256')==h and trial.get('model_signature')==ASR_SIGNATURE:
                    rec=trial
            if rec is None:
                staged=self.stage/(h+'.wav')
                if not staged.exists() or sha_file(staged)!=h:shutil.copyfile(p,staged)
                if sha_file(staged)!=h:raise IOError('ASR input hash mismatch')
                # One audio per call makes result-to-file association unambiguous.
                with torch.inference_mode(False), torch.no_grad():
                    hyp=self.model.transcribe([str(staged)],return_hypotheses=True)
                text=hyp_text(hyp)
                rec=dict(text=text,wav_sha256=h,model_signature=ASR_SIGNATURE,
                         staged_name=staged.name,source_path=str(p))
                put_json(dest,rec)
            if not isinstance(rec.get('text'),str):raise ValueError('Invalid cached transcript')
            result[cid]=rec
        return result
ASR=FreshASR(asr_model)

def text_tokens(text):
    text=re.sub(r'<[^>]*>|\[[^\]]*\]',' ',text)
    text=unicodedata.normalize('NFC',text).lower()
    return ''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text).split()

def word_counts(ref,hyp):
    # Deterministic minimum-edit alignment; ties prefer substitution, then deletion.
    a,b=text_tokens(ref),text_tokens(hyp);n,m=len(a),len(b)
    dp=np.zeros((n+1,m+1),dtype=np.int32)
    dp[:,0]=np.arange(n+1);dp[0,:]=np.arange(m+1)
    for i in range(1,n+1):
        for j in range(1,m+1):
            dp[i,j]=min(dp[i-1,j-1]+(a[i-1]!=b[j-1]),dp[i-1,j]+1,dp[i,j-1]+1)
    i,j=n,m;sub=delete=insert=hit=0
    while i or j:
        if i and j and a[i-1]==b[j-1] and dp[i,j]==dp[i-1,j-1]:hit+=1;i-=1;j-=1
        elif i and j and dp[i,j]==dp[i-1,j-1]+1:sub+=1;i-=1;j-=1
        elif i and dp[i,j]==dp[i-1,j]+1:delete+=1;i-=1
        else:insert+=1;j-=1
    return dict(words=n,substitutions=sub,deletions=delete,insertions=insert,
                errors=sub+delete+insert,hits=hit)
assert word_counts('a b','a c')['substitutions']==1
assert word_counts('a b','a')['deletions']==1
assert word_counts('a','a b')['insertions']==1

BASELINES={}
def get_reference_decodes(label,rows):
    if label in BASELINES:return BASELINES[label]
    raw={r.clip_id:Path(r.noisy_path) for r in rows.itertuples()}
    raw_h=ASR.transcribe(raw)
    clean_h=ASR.transcribe({r.clip_id:Path(r.clean_path) for r in rows.itertuples()}) if 'clean_path' in rows else None
    BASELINES[label]=(raw_h,clean_h)
    return raw_h,clean_h
get_reference_decodes('synthetic_dev',SYN_DEV)
get_reference_decodes('natural_dev',NAT_DEV)
# Independent uncached re-decode of one noisy/clean pair.
first=SYN_DEV.iloc[0]
check=ASR.transcribe({'noisy':Path(first.noisy_path),'clean':Path(first.clean_path)},force=True)
put_json(RUN/'uncached_pair_check.json',check)
print('First pair fresh decode:',{k:v['text'] for k,v in check.items()})


In [ ]:
def process_rows(rows,directory,checkpoint_sha):
    directory.mkdir(parents=True,exist_ok=True);paths={};audit=[]
    for r in tqdm(rows.to_dict('records'),desc='Always-on enhancement'):
        cid=r['clip_id'];x=read_wave_strict(r['noisy_path'])
        p=directory/(cid+'.wav');rec_path=directory/(cid+'.audit.json')
        key=sha_json(dict(input=hashlib.sha256(x.tobytes()).hexdigest(),checkpoint=checkpoint_sha,impl=IMPLEMENTATION))
        rec=None
        if p.exists() and rec_path.exists():
            trial=json.loads(rec_path.read_text())
            if trial.get('key')==key and trial.get('wav_sha256')==wav_audit(p,len(x)):rec=trial
        if rec is None:
            z=enhance_audio(x);write_pcm(p,z)
            rec=dict(clip_id=cid,key=key,model_calls=LAST_MODEL_CALLS,wav_sha256=wav_audit(p,len(x)),
                unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
                clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768))),
                change_rms=float(np.sqrt(np.mean((pcm_roundtrip(z)-pcm_roundtrip(x))**2))))
            put_json(rec_path,rec)
        paths[cid]=p;audit.append(rec)
    return paths,audit

def evaluate(label,rows,paths,candidate):
    raw,clean=get_reference_decodes(label,rows)
    enh=ASR.transcribe(paths);details=[];skipped=0
    for r in rows.to_dict('records'):
        cid=r['clip_id'];ref=r['transcript']
        if not text_tokens(ref):skipped+=1;continue
        record=dict(clip_id=cid,reference=ref,raw_text=raw[cid]['text'],enhanced_text=enh[cid]['text'],
            raw_sha=raw[cid]['wav_sha256'],enhanced_sha=enh[cid]['wav_sha256'],
            same_transcript=text_tokens(raw[cid]['text'])==text_tokens(enh[cid]['text']))
        for role,h in [('raw',raw),('enhanced',enh),('clean',clean)]:
            if h is not None:
                record.update({role+'_'+k:v for k,v in word_counts(ref,h[cid]['text']).items()})
                if role=='clean':record['clean_text']=h[cid]['text'];record['clean_sha']=h[cid]['wav_sha256']
        if 'clean_path' in r:
            s=read_wave_strict(r['clean_path']);x=read_wave_strict(r['noisy_path']);z=read_wave_strict(paths[cid])
            record['noisy_si_sdr']=si_sdr(x,s);record['enhanced_si_sdr']=si_sdr(z,s)
        record['error_increase']=record['enhanced_errors']-record['raw_errors']
        details.append(record)
    if not details:raise ValueError('No usable reference transcripts')
    frame=pd.DataFrame(details).sort_values('error_increase',ascending=False)
    frame.to_csv(RUN/(candidate+'_'+label+'_details.csv'),index=False)
    words=int(frame.raw_words.sum());summary=dict(candidate=candidate,subset=label,clips=len(frame),skipped=skipped,reference_words=words,raw_errors=int(frame.raw_errors.sum()),enhanced_errors=int(frame.enhanced_errors.sum()),
        raw_wer_percent=100*frame.raw_errors.sum()/words,
        enhanced_wer_percent=100*frame.enhanced_errors.sum()/words,
        identical_transcript_fraction=float(frame.same_transcript.mean()))
    summary['delta_wer_points']=summary['raw_wer_percent']-summary['enhanced_wer_percent']
    for role in ['raw','enhanced']:
        for typ in ['substitutions','deletions','insertions']:summary[role+'_'+typ]=int(frame[role+'_'+typ].sum())
    if clean is not None:
        summary['clean_wer_percent']=100*frame.clean_errors.sum()/words
        summary['enhanced_si_sdr']=float(frame.enhanced_si_sdr.mean())
        summary['noisy_si_sdr']=float(frame.noisy_si_sdr.mean())
    print(summary);return summary



In [ ]:
summaries=[];selection=[]
for candidate,checkpoint in CANDIDATES.items():
    asr_model.to("cpu");torch.cuda.empty_cache();activate(checkpoint);checkpoint_sha=sha_file(checkpoint)
    syn_paths,syn_audit=process_rows(SYN_DEV,WORK/candidate/"synthetic_dev",checkpoint_sha)
    nat_paths,nat_audit=process_rows(NAT_DEV,WORK/candidate/"natural_dev",checkpoint_sha)
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
    a=evaluate("synthetic_dev",SYN_DEV,syn_paths,candidate);b=evaluate("natural_dev",NAT_DEV,nat_paths,candidate)
    summaries.extend([a,b]);words=a["reference_words"]+b["reference_words"]
    dwer=100*((a["raw_errors"]+b["raw_errors"])-(a["enhanced_errors"]+b["enhanced_errors"]))/words
    proxy=a["enhanced_si_sdr"]+dwer
    unchanged=any(r["unchanged_pcm"] for r in syn_audit+nat_audit)
    selection.append(dict(candidate=candidate,synthetic_si_sdr=a["enhanced_si_sdr"],pooled_delta_wer_points=dwer,
                          combined_proxy=proxy,unchanged=unchanged))
    pd.DataFrame(syn_audit+nat_audit).to_csv(RUN/(candidate+"_processing_audit.csv"),index=False)
    put_json(RUN/"development_metrics.json",summaries)
eligible_sel=[r for r in selection if not r["unchanged"]]
if not eligible_sel:raise RuntimeError("No genuinely processing checkpoint")
BEST=max(eligible_sel,key=lambda r:r["combined_proxy"])["candidate"]
SELECTED=CANDIDATES[BEST];SELECTED_SHA=sha_file(SELECTED)
put_json(RUN/"selected_model.json",dict(candidate=BEST,checkpoint=str(SELECTED),sha256=SELECTED_SHA,scores=selection,
         criterion="synthetic SI-SDR plus pooled WER improvement points"))
display(pd.DataFrame(selection).sort_values("combined_proxy",ascending=False));print("Selected:",BEST)

# Monitoring is reported once and never used to reselect among candidates.
asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
sp,sa=process_rows(SYN_MON,WORK/BEST/"synthetic_monitor",SELECTED_SHA)
npth,na=process_rows(NAT_MON,WORK/BEST/"natural_monitor",SELECTED_SHA)
model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
monitor=[evaluate("synthetic_monitor",SYN_MON,sp,BEST),evaluate("natural_monitor",NAT_MON,npth,BEST)]
put_json(RUN/"monitor_metrics.json",monitor)


## 11 — Optional Phase-2 submission ZIP


In [ ]:
if RUN_PHASE2:
    if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:
        raise RuntimeError("Rules confirmation is required before submission generation")
    ids=sorted(TRANSCRIPT_MAP)
    if len(ids)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected Phase-2 clip count")
    source=StrictWaveSource(SOURCE_AUDIO,ids)
    asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
    wavs=WORK/"phase2"/SELECTED_SHA[:12];wavs.mkdir(parents=True,exist_ok=True)
    paths={};frames={};audits=[]
    for cid in tqdm(ids,desc="All-clip always-on enhancement"):
        x=source.load(cid);z=enhance_audio(x);p=wavs/(cid+".wav");write_pcm(p,z)
        audits.append(dict(clip_id=cid,input_sha256=hashlib.sha256(x.tobytes()).hexdigest(),
            wav_sha256=wav_audit(p,len(x)),checkpoint_sha256=SELECTED_SHA,model_calls=LAST_MODEL_CALLS,
            unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
            clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768)))))
        paths[cid]=p;frames[cid]=len(x)
    source.close();pd.DataFrame(audits).to_csv(RUN/"phase2_processing_audit.csv",index=False)
    if any(r["unchanged_pcm"] for r in audits):raise RuntimeError("Unchanged output found; no bypass permitted")
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval();transcripts=ASR.transcribe(paths)
    ZIP=RUN/("submission_Track2_PHASE2_V32_IndicParler_BEATs_Demucs_"+BEST+"_"+SELECTED_SHA[:10]+".zip")
    package=pack_submission(paths,transcripts,frames,ZIP)
    put_json(RUN/"submission_manifest.json",dict(version="v32_indicparler_beats_demucs_finetune",candidate=BEST,
        checkpoint_sha256=SELECTED_SHA,all_clips_enhanced=True,routing=False,dictionary_inference=False,
        beats_in_inference=False,tts_in_inference=False,pseudo_pairs=PSEUDO_PAIRS,
        source_derived_training_permission_confirmed=True,package=package))
    print("ZIP ready for manual upload:",ZIP)
else:
    print("RUN_PHASE2=False: metrics generated; no submission ZIP created.")


## Outputs and interpretation

- `noise_profiles_train.csv`: isolated residual bank from training-split validation pairs.
- `source_events/*.json`: resumable BEATs+WavLM boundary estimates.
- `source_profile_matches/*.json`: BEATs semantic top-five profile matches.
- `parler_clean_16k/*.wav`: cached Indic Parler clean targets.
- `pseudo_pairs.jsonl`: exactly 5,000 aligned pseudo-pair records.
- `demucs_epoch_*.pt`: resumable pretrained-Demucs fine-tuning candidates.
- `development_metrics.json`, `monitor_metrics.json`, `selected_model.json`: checkpoint evidence.
- `phase2_processing_audit.csv`: proof that every submitted clip was processed.

Passing the mechanical audit does not establish competition-rule compliance. Keep the source-derived-training flag false unless the organizers explicitly permit using evaluation-derived transcripts or features for fitting. The leave-out real development and monitoring results—not pseudo validation loss—are the decision evidence.
